
# MMF1921 - Backtesting & Full Results

# 1. Read input files

Same initialization as `Main.ipynb`

In [1]:
import pandas as pd
import numpy as np

from services.backtest import *
from services.project_function import *

adjClose = pd.read_csv("MMF1921_AssetPrices_1.csv", index_col=0)
factorRet = pd.read_csv("MMF1921_FactorReturns_1.csv", index_col=0)

adjClose.index = pd.to_datetime(adjClose.index)
factorRet.index = pd.to_datetime(factorRet.index)

In [2]:
print(adjClose.index.to_series().diff().value_counts())

Date
31 days    105
30 days     60
28 days     11
29 days      4
Name: count, dtype: int64


# 2. Perform Hyperparameter Tuning

In [3]:
grids = {

    "ridge_lw": {
        "NumObs": [36, 48, 60],
        "ridge_alpha": [0.01, 0.1, 1.0],
        "lw_shrink_weight": [0.3, 0.5, 0.7],
        "turnover_penalty": [0.0, 0.0001, 0.001, 0.005, 0.01],
    },

    "ols_mvo": {
        "NumObs": [24, 36, 48, 60],
        "risk_aversion": [1, 3, 5, 10],
        "max_weight": [0.10, 0.15, 0.20, 0.25],
    },

    "historical_mvo": {
        "NumObs": [24, 36, 48, 60],
        "risk_aversion": [1, 3, 5, 10],
        "max_weight": [0.10, 0.15, 0.20, 0.25],
    },

    "risk_parity": {
        "NumObs": [24, 36, 48, 60],
        "covariance_method": ["sample", "ledoit_wolf"],
    },

    "historical_max_sharpe": {
        "NumObs": [42, 48, 54],
        "max_weight": [0.05, 0.075, 0.10, 0.125],
    },
}

In [4]:
best_models = {}

for strategy_name, param_grid in grids.items():

    print("\n" + "="*60)
    print(f"Tuning {strategy_name}")
    print("="*60)

    best_params, test_summary = grid_search(
        adjClose,
        factorRet,
        strategy_name=strategy_name,
        param_grid=param_grid,
        turnover_weight=0.02
    )

    best_models[strategy_name] = {
        "params": best_params,
        "summary": test_summary
    }


Tuning ridge_lw
Strategy: ridge_lw
Running grid search over 135 combinations...

Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.3, 'turnover_penalty': 0.0} -> Sharpe: 0.4260, Turnover: 0.7757, Score: 0.4105
Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.3, 'turnover_penalty': 0.0001} -> Sharpe: 0.4259, Turnover: 0.7468, Score: 0.4109
Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.3, 'turnover_penalty': 0.001} -> Sharpe: 0.3880, Turnover: 0.5296, Score: 0.3774
Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.3, 'turnover_penalty': 0.005} -> Sharpe: 0.3205, Turnover: 0.2517, Score: 0.3155
Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.3, 'turnover_penalty': 0.01} -> Sharpe: 0.4042, Turnover: 0.1851, Score: 0.4005
Params: {'NumObs': 36, 'ridge_alpha': 0.01, 'lw_shrink_weight': 0.5, 'turnover_penalty': 0.0} -> Sharpe: 0.4172, Turnover: 0.7949, Score: 0.4013
Params: {'NumObs': 36, 'ridge_alpha': 0.

In [5]:
T = len(adjClose)

train_end = int(0.60 * T)
val_end = int(0.80 * T)

print("Train end index:", train_end)
print("Validation end index:", val_end)

print("Train start:", adjClose.index[0])
print("Train end:", adjClose.index[train_end-1])

print("Validation start:", adjClose.index[train_end])
print("Validation end:", adjClose.index[val_end-1])

print("Test start:", adjClose.index[val_end])
print("Test end:", adjClose.index[-1])

Train end index: 108
Validation end index: 144
Train start: 2001-12-31 00:00:00
Train end: 2010-11-30 00:00:00
Validation start: 2010-12-31 00:00:00
Validation end: 2013-11-30 00:00:00
Test start: 2013-12-31 00:00:00
Test end: 2016-12-31 00:00:00


# 3. Compare Models

In [6]:
best_models

{'ridge_lw': {'params': {'NumObs': 48,
   'ridge_alpha': 0.1,
   'lw_shrink_weight': 0.7,
   'turnover_penalty': 0.0},
  'summary': {'sharpe': 0.17958450326907166,
   'avg_turnover': 0.5226426074784751}},
 'ols_mvo': {'params': {'NumObs': 48, 'risk_aversion': 5, 'max_weight': 0.1},
  'summary': {'sharpe': 0.22460376339218857,
   'avg_turnover': 0.4098290527999023}},
 'historical_mvo': {'params': {'NumObs': 48,
   'risk_aversion': 5,
   'max_weight': 0.1},
  'summary': {'sharpe': 0.22392951114651236, 'avg_turnover': 0.4298398736908}},
 'risk_parity': {'params': {'NumObs': 36, 'covariance_method': 'sample'},
  'summary': {'sharpe': 0.28650544701713804,
   'avg_turnover': 0.17300902075663843}},
 'historical_max_sharpe': {'params': {'NumObs': 48, 'max_weight': 0.1},
  'summary': {'sharpe': 0.3220185498972464,
   'avg_turnover': 0.2924236142499972}}}

In [7]:
# 6. Build final comparison table using best parameters

strategies_to_compare = {
    "Ridge + LW": (
        "ridge_lw",
        best_models["ridge_lw"]["params"]
    ),

    "OLS MVO": (
        "ols_mvo",
        best_models["ols_mvo"]["params"]
    ),

    "Historical MVO": (
        "historical_mvo",
        best_models["historical_mvo"]["params"]
    ),

    "Risk Parity": (
        "risk_parity",
        best_models["risk_parity"]["params"]
    ),

    "Historical Max Sharpe": (
        "historical_max_sharpe",
        best_models["historical_max_sharpe"]["params"]
    ),

    "Equal Weight": (
        "equal_weight",
        {}
    ),
}

comparison_results = compare_strategies(
    adjClose,
    factorRet,
    strategies=strategies_to_compare
)

comparison_results


--- Ridge + LW ---

--- OLS MVO ---

--- Historical MVO ---

--- Risk Parity ---

--- Historical Max Sharpe ---

--- Equal Weight ---

=== Strategy Comparison ===
                       Sharpe  Avg Turnover
Strategy                                   
Ridge + LW             0.1991        0.5213
OLS MVO                0.1546        0.4590
Historical MVO         0.1606        0.4654
Risk Parity            0.1518        0.1788
Historical Max Sharpe  0.1750        0.3850
Equal Weight           0.1653        0.1170


,Sharpe,Avg Turnover
Strategy,,
Ridge + LW,0.1991,0.5213
OLS MVO,0.1546,0.4590
Historical MVO,0.1606,0.4654
Risk Parity,0.1518,0.1788
Historical Max Sharpe,0.1750,0.3850
Equal Weight,0.1653,0.1170


In [8]:
# 7. Add annualized Sharpe for interpretation

comparison_results["Annualized Sharpe"] = comparison_results["Sharpe"] * np.sqrt(12)

comparison_results

,Sharpe,Avg Turnover,Annualized Sharpe
Strategy,,,
Ridge + LW,0.1991,0.5213,0.689703
OLS MVO,0.1546,0.4590,0.535550
Historical MVO,0.1606,0.4654,0.556335
Risk Parity,0.1518,0.1788,0.525851
Historical Max Sharpe,0.1750,0.3850,0.606218
Equal Weight,0.1653,0.1170,0.572616


In [9]:
print(best_models["historical_max_sharpe"])

{'params': {'NumObs': 48, 'max_weight': 0.1}, 'summary': {'sharpe': 0.3220185498972464, 'avg_turnover': 0.2924236142499972}}


In [10]:
print(comparison_results)

                       Sharpe  Avg Turnover  Annualized Sharpe
Strategy                                                      
Ridge + LW             0.1991        0.5213           0.689703
OLS MVO                0.1546        0.4590           0.535550
Historical MVO         0.1606        0.4654           0.556335
Risk Parity            0.1518        0.1788           0.525851
Historical Max Sharpe  0.1750        0.3850           0.606218
Equal Weight           0.1653        0.1170           0.572616


In [11]:
comparison_results["Sharpe Rank"] = comparison_results["Sharpe"].rank(ascending=False)
comparison_results["Turnover Rank"] = comparison_results["Avg Turnover"].rank(ascending=True)

comparison_results["Combined Rank"] = (
    0.8 * comparison_results["Sharpe Rank"] +
    0.2 * comparison_results["Turnover Rank"]
)

comparison_results.sort_values("Combined Rank")

,Sharpe,Avg Turnover,Annualized Sharpe,Sharpe Rank,Turnover Rank,Combined Rank
Strategy,,,,,,
Ridge + LW,0.1991,0.5213,0.689703,1.0,6.0,2.0
Historical Max Sharpe,0.1750,0.3850,0.606218,2.0,3.0,2.2
Equal Weight,0.1653,0.1170,0.572616,3.0,1.0,2.6
Historical MVO,0.1606,0.4654,0.556335,4.0,5.0,4.2
OLS MVO,0.1546,0.4590,0.535550,5.0,4.0,4.8
Risk Parity,0.1518,0.1788,0.525851,6.0,2.0,5.2
